# Weryfikacja zapisu Parquet – PCR Lab Data

Sprawdzamy, czy pliki `pcr_events_biz.parquet` i `pcr_cases.parquet` wczytują się poprawnie i nie zawierają niespójności.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('/mnt/adata-disk/projects/agh/2stopien/BusinessModeling')
P_EVENTS = BASE / 'pcr_events_biz.parquet'
P_CASES  = BASE / 'pcr_cases.parquet'

print(f'pcr_events_biz.parquet : {P_EVENTS.stat().st_size / 1024:.1f} KB')
print(f'pcr_cases.parquet      : {P_CASES.stat().st_size  / 1024:.1f} KB')

pcr_events_biz.parquet : 1702.8 KB
pcr_cases.parquet      : 411.0 KB


## 1. Wczytanie plików

In [2]:
df_events = pd.read_parquet(P_EVENTS)
df_cases  = pd.read_parquet(P_CASES)

print(f'df_events : {df_events.shape[0]:,} wierszy × {df_events.shape[1]} kolumn')
print(f'df_cases  : {df_cases.shape[0]:,} wierszy × {df_cases.shape[1]} kolumn')

df_events : 134,099 wierszy × 12 kolumn
df_cases  : 6,339 wierszy × 7 kolumn


## 2. Typy kolumn i przykładowe dane

In [3]:
print('=== df_events – typy ===')
print(df_events.dtypes.to_string())
print()
print('=== df_cases – typy ===')
print(df_cases.dtypes.to_string())

=== df_events – typy ===
case_id                           int64
instance_uuid                    object
case_name                        object
activity                         object
endpoint                         object
lifecycle                        object
cpee_lifecycle                   object
cpee_activity_id                 object
cpee_state                       object
timestamp           datetime64[ns, UTC]
result                           object
ct                              float64

=== df_cases – typy ===
instance_uuid                 object
n_events                       int64
n_activities                   int64
first_ts         datetime64[ns, UTC]
last_ts          datetime64[ns, UTC]
duration_min                 float64
pcr_result                    object


In [4]:
print('=== df_events – pierwsze wiersze ===')
df_events.head()

=== df_events – pierwsze wiersze ===


,case_id,instance_uuid,case_name,activity,endpoint,lifecycle,cpee_lifecycle,cpee_activity_id,cpee_state,timestamp,result,ct
0,15143,003effb8-cb6d-468d-a243-a12763e5021c,Sample: 0cca00f1-183f-44a0-a132-b29ec7f34f6e,timeout,https://mygreschner.com/backend/services/timeout2,start,activity/calling,a2,None,2023-05-02 14:17:41.174000+00:00,None,NaN
1,15143,003effb8-cb6d-468d-a243-a12763e5021c,Sample: 0cca00f1-183f-44a0-a132-b29ec7f34f6e,Match patient data,https://mygreschner.com//backend/corr,start,activity/calling,a4,None,2023-05-02 14:17:41.185000+00:00,None,NaN
2,15143,003effb8-cb6d-468d-a243-a12763e5021c,Sample: 0cca00f1-183f-44a0-a132-b29ec7f34f6e,Wait for plate validation,https-get://cpee.org/ing/correlators/message/r...,start,activity/calling,a6,None,2023-05-02 14:17:41.186000+00:00,None,NaN
3,15143,003effb8-cb6d-468d-a243-a12763e5021c,Sample: 0cca00f1-183f-44a0-a132-b29ec7f34f6e,Match patient data,https://mygreschner.com//backend/corr,complete,activity/done,a4,None,2023-05-02 14:17:47.950000+00:00,None,NaN
4,15143,003effb8-cb6d-468d-a243-a12763e5021c,Sample: 0cca00f1-183f-44a0-a132-b29ec7f34f6e,Wait for plate validation,https-get://cpee.org/ing/correlators/message/r...,complete,activity/done,a6,None,2023-05-02 16:45:19.629000+00:00,None,NaN


In [5]:
print('=== df_cases – pierwsze wiersze ===')
df_cases.head()

=== df_cases – pierwsze wiersze ===


,instance_uuid,n_events,n_activities,first_ts,last_ts,duration_min,pcr_result
0,003effb8-cb6d-468d-a243-a12763e5021c,10,5,2023-05-02 14:17:41.174000+00:00,2023-05-02 16:45:25.938000+00:00,147.746067,NEGATIVE
1,0040da01-8f4a-426a-b632-5582ef74f0b2,10,5,2023-04-20 13:53:03.420000+00:00,2023-04-20 16:42:07.852000+00:00,169.073867,POSITIVE
2,004308fd-62f1-4e05-9d0c-0a22acade894,10,5,2023-04-14 16:51:06.848000+00:00,2023-04-14 19:38:36.215000+00:00,167.489450,POSITIVE
3,00513cde-4692-43cd-85aa-d8422ba79a76,10,5,2023-04-18 17:21:29.544000+00:00,2023-04-18 20:09:35.598000+00:00,168.100900,NEGATIVE
4,0052a6bd-c098-4e11-8944-46e44ec37ae9,9,5,2023-04-04 12:57:06.435000+00:00,2023-04-04 20:05:45.470000+00:00,428.650583,NEGATIVE


## 3. Sprawdzenie typów mieszanych (problem z parquetem)

In [12]:
print('Sprawdzanie kolumn object pod kątem mieszanych typów...')
issues = []
for col in df_events.select_dtypes(include='object').columns:
    types = df_events[col].dropna().map(type).value_counts()
    if len(types) > 1:
        issues.append((col, types.to_dict()))
        print(f'!!! - {col}: {types.to_dict()}')

if not issues:
    print('Brak kolumn z mieszanymi typami w df_events')

print()
issues2 = []
for col in df_cases.select_dtypes(include='object').columns:
    types = df_cases[col].dropna().map(type).value_counts()
    if len(types) > 1:
        issues2.append((col, types.to_dict()))
        print(f'!!! - {col}: {types.to_dict()}')

if not issues2:
    print('Brak kolumn z mieszanymi typami w df_cases')

Sprawdzanie kolumn object pod kątem mieszanych typów...
Brak kolumn z mieszanymi typami w df_events

Brak kolumn z mieszanymi typami w df_cases


## 4. Sprawdzenie wyników PCR – mapowanie N/P

In [13]:
print('=== df_events["result"] – unikalne wartości ===')
print(df_events['result'].value_counts(dropna=False).to_string())
print()
print('=== df_cases["pcr_result"] – unikalne wartości ===')
print(df_cases['pcr_result'].value_counts(dropna=False).to_string())

# Sprawdź czy nie ma już surowych dictów ani krótkich kodów
bad_events = df_events[df_events['result'].isin(['N', 'P'])]
bad_cases  = df_cases[df_cases['pcr_result'].isin(['N', 'P'])]
print(f'\nWiersze z result="N" lub "P" (powinno być 0): events={len(bad_events)}, cases={len(bad_cases)}')

if len(bad_events) == 0 and len(bad_cases) == 0:
    print('Mapowanie N→NEGATIVE / P->POSITIVE działa poprawnie')
else:
    print('!!! - Pozostały niezamapowane wartości!')

=== df_events["result"] – unikalne wartości ===
result
None    134099

=== df_cases["pcr_result"] – unikalne wartości ===
pcr_result
NEGATIVE    3174
POSITIVE    2818
None         346
IPC            1

Wiersze z result="N" lub "P" (powinno być 0): events=0, cases=0
Mapowanie N→NEGATIVE / P->POSITIVE działa poprawnie


## 5. Sprawdzenie timestampów

In [14]:
ts_col = df_events['timestamp']
print(f'Typ kolumny timestamp : {ts_col.dtype}')
print(f'Brakujące timestamps  : {ts_col.isna().sum():,}')
print(f'Zakres: {ts_col.min()} → {ts_col.max()}')

# Sprawdź ujemne delty (timestamp wsteczny w obrębie jednego case)
df_sorted = df_events.sort_values(['instance_uuid', 'timestamp'])
df_sorted['ts_delta'] = df_sorted.groupby('instance_uuid')['timestamp'].diff().dt.total_seconds()
neg_deltas = df_sorted[df_sorted['ts_delta'] < 0]
print(f'\nEventy z ujemną deltą czasową (wsteczne): {len(neg_deltas):,}')
if len(neg_deltas) > 0:
    print(neg_deltas[['instance_uuid', 'activity', 'timestamp', 'ts_delta']].head(5).to_string())

Typ kolumny timestamp : datetime64[ns, UTC]
Brakujące timestamps  : 0
Zakres: 2023-04-02 16:00:01.367000+00:00 → 2023-06-14 07:24:15.779000+00:00

Eventy z ujemną deltą czasową (wsteczne): 0


## 6. Spójność między df_events a df_cases

In [15]:
cases_in_events = set(df_events['instance_uuid'].unique())
cases_in_cases  = set(df_cases['instance_uuid'].unique())

only_events = cases_in_events - cases_in_cases
only_cases  = cases_in_cases - cases_in_events

print(f'Cases w df_events : {len(cases_in_events):,}')
print(f'Cases w df_cases  : {len(cases_in_cases):,}')
print(f'Tylko w df_events (brak w df_cases) : {len(only_events)}')
print(f'Tylko w df_cases  (brak w df_events): {len(only_cases)}')

if not only_events and not only_cases:
    print('Oba pliki zawierają te same cases')
else:
    print('!!! - Rozbieżność – sprawdź cases bez eventów biznesowych')

Cases w df_events : 6,339
Cases w df_cases  : 6,339
Tylko w df_events (brak w df_cases) : 0
Tylko w df_cases  (brak w df_events): 0
Oba pliki zawierają te same cases


## 7. Podstawowa integralność danych

In [16]:
print('=== Braki w kluczowych kolumnach df_events ===')
for col in ['instance_uuid', 'activity', 'timestamp', 'lifecycle']:
    n = df_events[col].isna().sum()
    pct = 100 * n / len(df_events)
    status = 'OK' if n == 0 else '!!!'
    print(f'  {status} {col:<20}: {n:>6,} braków ({pct:.1f}%)')

print()
print('=== Dopuszczalne wartości lifecycle ===')
print(df_events['lifecycle'].value_counts(dropna=False).to_string())

print()
print('=== Duplikaty (instance_uuid + activity + lifecycle + timestamp) ===')
dups = df_events.duplicated(subset=['instance_uuid', 'activity', 'lifecycle', 'timestamp']).sum()
print(f'  {"OK" if dups == 0 else "!!!"} Duplikatów: {dups}')

=== Braki w kluczowych kolumnach df_events ===
  OK instance_uuid       :      0 braków (0.0%)
  OK activity            :      0 braków (0.0%)
  OK timestamp           :      0 braków (0.0%)
  OK lifecycle           :      0 braków (0.0%)

=== Dopuszczalne wartości lifecycle ===
lifecycle
complete    82890
start       51209

=== Duplikaty (instance_uuid + activity + lifecycle + timestamp) ===
  OK Duplikatów: 0


## 8. Podsumowanie weryfikacji

In [18]:
print('=' * 50)
print('PODSUMOWANIE WERYFIKACJI')
print('=' * 50)
print(f'  df_events : {len(df_events):,} eventów, {df_events["instance_uuid"].nunique():,} cases')
print(f'  df_cases  : {len(df_cases):,} cases')
print(f'  Zakres dat: {df_events["timestamp"].min().date()} → {df_events["timestamp"].max().date()}')
print(f'  Wyniki PCR: {dict(df_cases["pcr_result"].value_counts(dropna=False))}')
print(f'  Kolumny mieszanych typów: {len(issues) + len(issues2)}')
print(f'  Złe kody N/P: {len(bad_events) + len(bad_cases)}')
print(f'  Rozbieżność cases events↔cases: {len(only_events) + len(only_cases)}')
print()
all_ok = (not issues) and (not issues2) and (len(bad_events)==0) and (len(bad_cases)==0) and (not only_events) and (not only_cases)
if all_ok:
    print('Dane zapisane poprawnie – gotowe do Milestone 2!')
else:
    print('!!! - Są problemy – sprawdź powyższe sekcje')

PODSUMOWANIE WERYFIKACJI
  df_events : 134,099 eventów, 6,339 cases
  df_cases  : 6,339 cases
  Zakres dat: 2023-04-02 → 2023-06-14
  Wyniki PCR: {'NEGATIVE': np.int64(3174), 'POSITIVE': np.int64(2818), None: np.int64(346), 'IPC': np.int64(1)}
  Kolumny mieszanych typów: 0
  Złe kody N/P: 0
  Rozbieżność cases events↔cases: 0

Dane zapisane poprawnie – gotowe do Milestone 2!
